# Lab 04 · File formats and data layout

**Part A** (about 50 min): write one dataset in five formats and several layouts, and measure what each choice costs.<br>
**Part B** (about 70 min): 13 problems with dates, text and arrays.

**Checklist questions Part A gives you evidence for:**<br>
<span style="color:#d93025"><b>2.1</b> Why CSV is heavy · <b>2.2</b> Columnar storage · <b>2.3</b> Schema evolution · <b>2.4</b> Predicate pushdown · <b>2.5</b> The small file problem · <b>2.6</b> Avro for streaming · <b>2.7</b> The disadvantage of Parquet · <b>2.8</b> Partitioned folders · <b>2.9</b> Splittable files · <b>2.10</b> Parquet vs. ORC</span>

## How to work
* Run the **Setup** cell first, then answer the questions in order. `F` is `pyspark.sql.functions`.
* **Part A**: run the code under the question (or add a line), look at the output, and write your answer as a comment. Everything is written under `data/shop/`, which the setup cell clears.
* **Part B**: write the code yourself. Your result must look exactly like the **expected output**, with the rows in the same order.
* The answers are collapsed at the very end of the notebook. Open one only after you have tried.

## Documentation
* [Data sources](https://spark.apache.org/docs/latest/sql-data-sources.html): CSV, JSON, Parquet, ORC and Avro, with their options
* [Parquet files](https://spark.apache.org/docs/latest/sql-data-sources-parquet.html): schema merging, filter pushdown
* [Avro files](https://spark.apache.org/docs/latest/sql-data-sources-avro.html)
* [Parquet file format](https://parquet.apache.org/docs/file-format/): row groups, column chunks, statistics
* [Generic file source options](https://spark.apache.org/docs/latest/sql-data-sources-generic-options.html)
* [Tuning: `spark.sql.files.maxPartitionBytes`](https://spark.apache.org/docs/latest/sql-performance-tuning.html#tuning-partitions)
* [Datetime patterns](https://spark.apache.org/docs/latest/sql-ref-datetime-pattern.html): `dd/MM/yyyy`, `MMMM`, ...
* [Functions](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/functions.html): `to_date`, `last_day`, `date_trunc`, `regexp_replace`, `split`, `explode`, `collect_set`, `array_contains`, `sequence`

## The data
Three small tables of an online shop. The setup cell creates them, and you can read every row in it.

* **customers**: *customer_id*, *name*, *email*, *city*, *signup_date* (the day the customer registered)
* **products**: *product_id*, *product_name*, *category*, *price* (dollars for one item)
* **orders**: *order_id*, *customer_id*, *product_id*, *quantity*, *order_date*, *status* (`delivered`, `shipped` or `cancelled`)

Good to know:
* order 1011 is a guest checkout: it has no *customer_id*
* customer 6 has no email and no city
* customer 7 has never ordered, and product 107 has never been sold
* order 1016 is for product 108, which is no longer in *products*

For this lab the setup cell also makes:
* **big_orders** (Part A): 1,000,000 generated orders with the same columns as *orders*. The values look random but are the same on every machine.
* **reviews** (Part B): *review_id*, *product_id*, *customer_id*, *review_date* (text like `15/01/2024`), *rating* (1 to 5), *comment* (messy text) and *tags* (text like `fast delivery;good price`, sometimes empty).

## Setup
Run this cell once. It starts Spark with Avro support, saves the three shop tables as Parquet files in `data/shop/`, and creates *big_orders* and *reviews*. The first start may print a list of packages: that is Spark loading the Avro library.

In [ ]:
# Setup: starts Spark and creates the shop tables. Run it once.
import logging, os, shutil, sys, time
from datetime import date
os.environ.setdefault("PYSPARK_PYTHON", sys.executable)   # Spark's Python workers use this notebook's Python
from pyspark.sql import SparkSession, functions as F

spark = (SparkSession.builder
         .master("local[4]")   # 4 tasks at a time on your machine
         .appName("lab04")
         .config("spark.sql.adaptive.enabled", "true")        # AQE on, as in Spark's default
         .config("spark.sql.session.timeZone", "UTC")
         .config("spark.jars.packages", "org.apache.spark:spark-avro_2.13:4.2.0")   # Avro support (question A1)
         .config("spark.ui.showConsoleProgress", "false")
         .config("spark.log.level", "FATAL")                  # hide Spark's log messages
         # Spark 4.2.0 bug (SPARK-60294): Python UDFs on Parquet can hang without this
         .config("spark.sql.execution.arrow.pythonUDF.columnarInput.enabled", "false")
         .getOrCreate())
for name in ("DataFrameQueryContextLogger", "SQLQueryContextLogger"):
    logging.getLogger(name).setLevel(logging.CRITICAL)        # errors still show, without an extra JSON log

customers_rows = [
    # customer_id, name, email, city, signup_date
    (1, "Aziz Karimov",     "aziz@mail.uz",     "Tashkent", date(2024, 1, 5)),
    (2, "Bella Schmidt",    "bella@web.de",     "Berlin",   date(2024, 1, 20)),
    (3, "Carlos Ruiz",      "carlos@correo.es", "Madrid",   date(2024, 2, 2)),
    (4, "Dilnoza Rahimova", "dilnoza@mail.uz",  "Tashkent", date(2024, 2, 14)),
    (5, "Emma Wilson",      "emma@post.co.uk",  "London",   date(2024, 3, 1)),
    (6, "Farid Aliyev",     None,               None,       date(2024, 3, 10)),
    (7, "Grace Lee",        "grace@web.de",     "Berlin",   date(2024, 3, 15)),
]
products_rows = [
    # product_id, product_name, category, price
    (101, "Laptop",       "Electronics", 1200.0),
    (102, "Headphones",   "Electronics",  150.0),
    (103, "Coffee Maker", "Kitchen",       85.0),
    (104, "Blender",      "Kitchen",       60.0),
    (105, "Novel",        "Books",         15.0),
    (106, "Cookbook",     "Books",         25.0),
    (107, "Desk Lamp",    "Home",          40.0),
]
orders_rows = [
    # order_id, customer_id, product_id, quantity, order_date, status
    (1001, 1,    101, 1, date(2024, 1, 10), "delivered"),
    (1002, 1,    105, 3, date(2024, 1, 10), "delivered"),
    (1003, 2,    102, 2, date(2024, 1, 25), "delivered"),
    (1004, 3,    103, 1, date(2024, 2, 3),  "cancelled"),
    (1005, 2,    106, 1, date(2024, 2, 5),  "delivered"),
    (1006, 4,    104, 2, date(2024, 2, 15), "shipped"),
    (1007, 1,    102, 1, date(2024, 2, 20), "delivered"),
    (1008, 5,    101, 1, date(2024, 3, 2),  "shipped"),
    (1009, 3,    105, 4, date(2024, 3, 5),  "delivered"),
    (1010, 4,    106, 2, date(2024, 3, 8),  "cancelled"),
    (1011, None, 104, 1, date(2024, 3, 9),  "delivered"),
    (1012, 6,    103, 2, date(2024, 3, 12), "shipped"),
    (1013, 2,    101, 1, date(2024, 3, 18), "delivered"),
    (1014, 5,    105, 5, date(2024, 3, 20), "delivered"),
    (1015, 1,    103, 1, date(2024, 3, 25), "shipped"),
    (1016, 3,    108, 1, date(2024, 3, 28), "delivered"),
]

# Write the tables as Parquet files and read them back, the way real data arrives.
# data/shop belongs to this lab: it is rebuilt every time you run this cell.
shutil.rmtree("data/shop", ignore_errors=True)
for table, rows, schema in [
    ("customers", customers_rows, "customer_id INT, name STRING, email STRING, city STRING, signup_date DATE"),
    ("products",  products_rows,  "product_id INT, product_name STRING, category STRING, price DOUBLE"),
    ("orders",    orders_rows,    "order_id INT, customer_id INT, product_id INT, quantity INT, order_date DATE, status STRING"),
]:
    spark.createDataFrame(rows, schema).coalesce(1).write.parquet(f"data/shop/{table}")

customers = spark.read.parquet("data/shop/customers")
products = spark.read.parquet("data/shop/products")
orders = spark.read.parquet("data/shop/orders")

def pick(salt, n):
    """A number from 0 to n-1 computed from the row id: looks random, but is the same on every machine."""
    return F.pmod(F.xxhash64("id", F.lit(salt)), F.lit(n))

# Part A: one million generated orders with the same columns as orders (nothing is written yet)
big_orders = (spark.range(1_000_000)
              .select((F.col("id") + 1).alias("order_id"),
                      (pick(1, 5000) + 1).alias("customer_id"),
                      (pick(2, 7) + 101).alias("product_id"),
                      (pick(3, 5) + 1).alias("quantity"),
                      F.date_add(F.lit("2024-01-01").cast("date"), pick(4, 91).cast("int")).alias("order_date"),
                      F.element_at(F.array(F.lit("delivered"), F.lit("shipped"), F.lit("cancelled")),
                                   (pick(5, 3) + 1).cast("int")).alias("status")))

def folder_mb(path):
    """Size of the data files in a folder (and its subfolders), in MB."""
    sizes = [os.path.getsize(os.path.join(d, f)) for d, _, files in os.walk(path) for f in files if not f.startswith((".", "_"))]
    return round(sum(sizes) / 1e6, 1)

def count_jobs(action):
    """Runs action() and returns how many Spark jobs it started."""
    tracker = spark.sparkContext.statusTracker()
    before = set(tracker.getJobIdsForGroup())
    action()
    time.sleep(0.5)                       # the Spark UI records a job a moment after it ends
    return len(set(tracker.getJobIdsForGroup()) - before)

def best_time(action, runs=3):
    """Runs action() a few times and returns the fastest time in seconds (the first run warms up)."""
    times = []
    for _ in range(runs):
        start = time.time()
        action()
        times.append(time.time() - start)
    return round(min(times), 2)

# Part B: product reviews
reviews = spark.createDataFrame([
    # review_id, product_id, customer_id, review_date (text), rating, comment, tags
    (1, 101, 1, "15/01/2024", 5, "  Great laptop, very FAST!  ", "fast delivery;good price"),
    (2, 105, 1, "20/01/2024", 4, "Nice story.",                  "good price"),
    (3, 102, 2, "02/02/2024", 3, "Sound is ok, delivery was slow", "slow delivery"),
    (4, 103, 3, "10/02/2024", 1, "Broke after one week!!",       "broken;slow delivery"),
    (5, 101, 5, "09/03/2024", 5, "Fast and powerful",            "fast delivery"),
    (6, 105, 3, "12/03/2024", 2, "Too long, boring",             None),
    (7, 106, 2, "25/03/2024", 4, "Useful recipes",               "good price;gift"),
    (8, 103, 6, "30/03/2024", 4, "Good coffee, slow start",      "gift"),
], "review_id INT, product_id INT, customer_id INT, review_date STRING, rating INT, comment STRING, tags STRING")

print("Spark", spark.version, "| Spark UI:", spark.sparkContext.uiWebUrl)

Look at the tables:

In [ ]:
customers.show()
products.show()
orders.show()
reviews.show(truncate=False)
big_orders.show(5)

## Part A · Formats and layouts

### Row formats and columnar formats <span style="color:#d93025">(2.1, 2.2, 2.10, 2.6)</span>

**A1. Write `big_orders` as CSV, JSON, Parquet, ORC and Avro. Which format is the biggest, and which the smallest?**

In [ ]:
# My answer:
for fmt in ["csv", "json", "parquet", "orc", "avro"]:
    big_orders.write.mode("overwrite").option("header", True).format(fmt).save(f"data/shop/formats/{fmt}")
    print(f"{fmt:8}", folder_mb(f"data/shop/formats/{fmt}"), "MB")

**A2. Read the CSV back without a schema. What type does every column get? And how many jobs does `inferSchema` start before you have even asked for any data?**

In [ ]:
# My answer:
spark.read.option("header", True).csv("data/shop/formats/csv").printSchema()

**A3. Sum the quantity of each format and time it with `best_time`. Which formats are the fastest to read?**

In [ ]:
# My answer:
for fmt in ["csv", "json", "parquet", "orc", "avro"]:
    df = spark.read.format(fmt).schema(big_orders.schema).option("header", True).load(f"data/shop/formats/{fmt}")
    print(f"{fmt:8}", best_time(lambda: df.agg(F.sum("quantity")).first()), "s")

**A4. Avro keeps the schema inside each file. Print the first bytes of one Avro file: what do you see?**

In [ ]:
# My answer:
import glob
avro_file = glob.glob("data/shop/formats/avro/*.avro")[0]
print(open(avro_file, "rb").read(400))

### Skipping data <span style="color:#d93025">(2.4, 2.8)</span>

**A5. Parquet stores the min and max of every column for each row group (a block of rows). Write `big_orders` once unsorted and once sorted by quantity. In each file, how many row groups can Spark skip for `quantity = 5`?**

In [ ]:
# My answer:
import glob
import pyarrow.parquet as pq
big_orders.coalesce(1).write.mode("overwrite").option("parquet.block.size", 256 * 1024).parquet("data/shop/unsorted")
(big_orders.orderBy("quantity").coalesce(1)
           .write.mode("overwrite").option("parquet.block.size", 256 * 1024).parquet("data/shop/sorted"))

def quantity_ranges(folder):
    """(min, max) of quantity in every row group of the folder's Parquet file, from the file footer."""
    meta = pq.ParquetFile(glob.glob(f"{folder}/*.parquet")[0]).metadata
    col = meta.schema.names.index("quantity")
    return [(meta.row_group(i).column(col).statistics.min, meta.row_group(i).column(col).statistics.max)
            for i in range(meta.num_row_groups)]

print(quantity_ranges("data/shop/unsorted")[:5])
print(quantity_ranges("data/shop/sorted")[:5])

**A6. Write `big_orders` partitioned by status, then filter `status = 'cancelled'`. What does the plan show under `PartitionFilters`, and how many of the files does Spark read?**

In [ ]:
# My answer:
big_orders.write.mode("overwrite").partitionBy("status").parquet("data/shop/by_status")
cancelled = spark.read.parquet("data/shop/by_status").filter(F.col("status") == "cancelled")

### File sizes and splitting <span style="color:#d93025">(2.5, 2.9)</span>

**A7. Write `big_orders` partitioned by order_date twice: after `repartition(50)` and after `repartition("order_date")`. How many files does each layout have, and how long does reading each take?**

In [ ]:
# My answer:
big_orders.repartition(50).write.mode("overwrite").partitionBy("order_date").parquet("data/shop/small_files")
big_orders.repartition("order_date").write.mode("overwrite").partitionBy("order_date").parquet("data/shop/one_file_per_day")

**A8. Write `big_orders` as one plain CSV file and as one gzip CSV file. With `spark.sql.files.maxPartitionBytes` set to 8 MB, how many read partitions does each file get?**

In [ ]:
# My answer:
big_orders.coalesce(1).write.mode("overwrite").option("header", True).csv("data/shop/csv_plain")
big_orders.coalesce(1).write.mode("overwrite").option("header", True).option("compression", "gzip").csv("data/shop/csv_gzip")
spark.conf.set("spark.sql.files.maxPartitionBytes", 8 * 1024 * 1024)      # 8 MB per read partition

### Schema evolution and updates <span style="color:#d93025">(2.3, 2.7)</span>

**A9. Write 5 orders without and 5 orders with a new column `discount` into two folders of `data/shop/evolve`. Read the parent folder with and without `mergeSchema`: which columns do you get?**

In [ ]:
# My answer:
big_orders.filter("order_id <= 5").write.mode("overwrite").parquet("data/shop/evolve/batch=1")
(big_orders.filter("order_id > 5 AND order_id <= 10").withColumn("discount", F.lit(0.1))
           .write.mode("overwrite").parquet("data/shop/evolve/batch=2"))

**A10. Parquet files can't be changed. To fix the status of a single order, how much data must you write?**

In [ ]:
# My answer:
fixed = (spark.read.parquet("data/shop/formats/parquet")
              .withColumn("status", F.when(F.col("order_id") == 5, "returned").otherwise(F.col("status"))))
fixed.write.mode("overwrite").parquet("data/shop/formats/parquet_fixed")
print(folder_mb("data/shop/formats/parquet"), "MB of data,", folder_mb("data/shop/formats/parquet_fixed"), "MB written to change one value")

**Done with Part A?** Explain each red checklist question at the top aloud, in your own words, using what you saw.

## Part B · Dates, text and arrays
13 problems on the shop tables and *reviews*. Where an expected output is left-aligned, it was printed with `.show(truncate=False)`, so that long text and arrays are not cut off: do the same.

**B1. Turn review_date (text like `15/01/2024`) into a real date column `review_day`. Show review_id, review_date and review_day.**

Expected output:

```
+---------+-----------+----------+
|review_id|review_date|review_day|
+---------+-----------+----------+
|        1| 15/01/2024|2024-01-15|
|        2| 20/01/2024|2024-01-20|
|        3| 02/02/2024|2024-02-02|
|        4| 10/02/2024|2024-02-10|
|        5| 09/03/2024|2024-03-09|
|        6| 12/03/2024|2024-03-12|
|        7| 25/03/2024|2024-03-25|
|        8| 30/03/2024|2024-03-30|
+---------+-----------+----------+
```

<details><summary>Hint</summary>

`F.to_date(column, "dd/MM/yyyy")`: the pattern describes how the text is written.

</details>

**B2. For the orders placed in January and February, show order_id, order_date, the last day of that month (`month_end`) and the days left until then (`days_left`).**

Expected output:

```
+--------+----------+----------+---------+
|order_id|order_date| month_end|days_left|
+--------+----------+----------+---------+
|    1001|2024-01-10|2024-01-31|       21|
|    1002|2024-01-10|2024-01-31|       21|
|    1003|2024-01-25|2024-01-31|        6|
|    1004|2024-02-03|2024-02-29|       26|
|    1005|2024-02-05|2024-02-29|       24|
|    1006|2024-02-15|2024-02-29|       14|
|    1007|2024-02-20|2024-02-29|        9|
+--------+----------+----------+---------+
```

<details><summary>Hint</summary>

`F.last_day(date)` and `F.datediff(end, start)`.

</details>

**B3. Count the orders per week: show week_start (the Monday of the week, as a date) and orders, for the weeks that have orders, in date order.**

Expected output:

```
+----------+------+
|week_start|orders|
+----------+------+
|2024-01-08|     2|
|2024-01-22|     1|
|2024-01-29|     1|
|2024-02-05|     1|
|2024-02-12|     1|
|2024-02-19|     1|
|2024-02-26|     1|
|2024-03-04|     3|
|2024-03-11|     1|
|2024-03-18|     2|
|2024-03-25|     2|
+----------+------+
```

<details><summary>Hint</summary>

`F.date_trunc("week", column)` moves a date back to its Monday (it returns a timestamp: cast it to a date).

</details>

**B4. Show the quarter and the month name of each customer's signup date: name, signup_date, quarter and month_name.**

Expected output:

```
+----------------+-----------+-------+----------+
|            name|signup_date|quarter|month_name|
+----------------+-----------+-------+----------+
|    Aziz Karimov| 2024-01-05|      1|   January|
|   Bella Schmidt| 2024-01-20|      1|   January|
|     Carlos Ruiz| 2024-02-02|      1|  February|
|Dilnoza Rahimova| 2024-02-14|      1|  February|
|     Emma Wilson| 2024-03-01|      1|     March|
|    Farid Aliyev| 2024-03-10|      1|     March|
|       Grace Lee| 2024-03-15|      1|     March|
+----------------+-----------+-------+----------+
```

<details><summary>Hint</summary>

`F.quarter(...)`; `F.date_format(date, "MMMM")` writes the full month name.

</details>

**B5. Clean the comments: remove the spaces at both ends, make them lower case, and remove every character that is not a letter, a digit or a space. Show review_id and clean_comment.**

Expected output:

```
+---------+-----------------------------+
|review_id|clean_comment                |
+---------+-----------------------------+
|1        |great laptop very fast       |
|2        |nice story                   |
|3        |sound is ok delivery was slow|
|4        |broke after one week         |
|5        |fast and powerful            |
|6        |too long boring              |
|7        |useful recipes               |
|8        |good coffee slow start       |
+---------+-----------------------------+
```

<details><summary>Hint</summary>

`F.trim`, `F.lower`, then `F.regexp_replace(column, "[^a-z0-9 ]", "")`.

</details>

**B6. How many words does each comment have? Show review_id and words.**

Expected output:

```
+---------+-----+
|review_id|words|
+---------+-----+
|        1|    4|
|        2|    2|
|        3|    6|
|        4|    4|
|        5|    3|
|        6|    3|
|        7|    2|
|        8|    4|
+---------+-----+
```

<details><summary>Hint</summary>

Split the trimmed text on spaces (`F.split(column, r"\s+")`) and count the pieces with `F.size`.

</details>

**B7. Which reviews mention "slow" in their comment, in any letter case? Show review_id and comment.**

Expected output:

```
+---------+------------------------------+
|review_id|comment                       |
+---------+------------------------------+
|3        |Sound is ok, delivery was slow|
|8        |Good coffee, slow start       |
+---------+------------------------------+
```

<details><summary>Hint</summary>

Lower-case the comment first, then `.contains("slow")` (or `.rlike("(?i)slow")`).

</details>

**B8. Split the tags into an array and show one row per review and tag (reviews without tags drop out). Show review_id and tag.**

Expected output:

```
+---------+-------------+
|review_id|          tag|
+---------+-------------+
|        1|fast delivery|
|        1|   good price|
|        2|   good price|
|        3|slow delivery|
|        4|       broken|
|        4|slow delivery|
|        5|fast delivery|
|        7|   good price|
|        7|         gift|
|        8|         gift|
+---------+-------------+
```

<details><summary>Hint</summary>

`F.split(column, ";")` makes an array; `F.explode` makes one row per element.

</details>

**B9. How often is each tag used? Show tag and reviews, most used first; on a tie, A to Z.**

Expected output:

```
+-------------+-------+
|          tag|reviews|
+-------------+-------+
|   good price|      3|
|fast delivery|      2|
|         gift|      2|
|slow delivery|      2|
|       broken|      1|
+-------------+-------+
```

<details><summary>Hint</summary>

Explode the tags like in B8, then group by tag.

</details>

**B10. For each customer, list the different products they ordered as an array of product names, sorted A to Z. Show customer_id and products.**

Expected output:

```
+-----------+-----------------------------------------+
|customer_id|products                                 |
+-----------+-----------------------------------------+
|1          |[Coffee Maker, Headphones, Laptop, Novel]|
|2          |[Cookbook, Headphones, Laptop]           |
|3          |[Coffee Maker, Novel]                    |
|4          |[Blender, Cookbook]                      |
|5          |[Laptop, Novel]                          |
|6          |[Coffee Maker]                           |
+-----------+-----------------------------------------+
```

<details><summary>Hint</summary>

`F.collect_set` gathers the distinct values of a group into an array; `F.array_sort` sorts it.

</details>

**B11. Collect each customer's product_ids into a sorted array, then keep only the customers whose array contains product 101. Show customer_id and product_ids.**

Expected output:

```
+-----------+--------------------+
|customer_id|product_ids         |
+-----------+--------------------+
|1          |[101, 102, 103, 105]|
|2          |[101, 102, 106]     |
|5          |[101, 105]          |
+-----------+--------------------+
```

<details><summary>Hint</summary>

`F.array_contains(array_column, 101)`.

</details>

**B12. Show the number of orders on every day from 2024-03-01 to 2024-03-10, with 0 on the days without orders. Columns: day and orders.**

Expected output:

```
+----------+------+
|       day|orders|
+----------+------+
|2024-03-01|     0|
|2024-03-02|     1|
|2024-03-03|     0|
|2024-03-04|     0|
|2024-03-05|     1|
|2024-03-06|     0|
|2024-03-07|     0|
|2024-03-08|     1|
|2024-03-09|     1|
|2024-03-10|     0|
+----------+------+
```

<details><summary>Hint</summary>

Build all the days with `F.sequence(start, end)` and `F.explode`, then left-join the counts and turn nulls into 0.

</details>

**B13. Mask the emails: keep the first letter, then `***`, then everything from the `@` on (like `a***@mail.uz`). Customers without an email drop out. Show name and masked_email.**

Expected output:

```
+----------------+---------------+
|            name|   masked_email|
+----------------+---------------+
|    Aziz Karimov|   a***@mail.uz|
|   Bella Schmidt|    b***@web.de|
|     Carlos Ruiz| c***@correo.es|
|Dilnoza Rahimova|   d***@mail.uz|
|     Emma Wilson|e***@post.co.uk|
|       Grace Lee|    g***@web.de|
+----------------+---------------+
```

<details><summary>Hint</summary>

`F.substring(column, 1, 1)`, `F.substring_index(column, "@", -1)` (the part after the last `@`) and `F.concat`.

</details>

---
**Done?** In the Theory part of Day 4, answer the red checklist questions aloud.

## Answers
Try each question yourself first. Click a question to open its answer; Part A answers also show the output of the reference run.

<details>
<summary><b>A1</b> &nbsp;Write big_orders as CSV, JSON, Parquet, ORC and Avro. Which format is the biggest, and which the smallest?</summary>

**JSON is the biggest, ORC the smallest, with Parquet close to it.** CSV and JSON are text: every value is written as characters, and JSON repeats every column name in every row. Parquet and ORC are **columnar**: they store all values of one column together, so similar values sit next to each other and compress very well (dictionary and run-length encoding), and they keep the types and min/max statistics. ORC and Parquet work the same way: ORC comes from Hive (built-in indexes, Hive ACID tables), Parquet is Spark's default and the base of Delta Lake. Avro is binary but **row-based**: one whole record after another.

```python
for fmt in ["csv", "json", "parquet", "orc", "avro"]:
    big_orders.write.mode("overwrite").option("header", True).format(fmt).save(f"data/shop/formats/{fmt}")
    print(f"{fmt:8}", folder_mb(f"data/shop/formats/{fmt}"), "MB")
```

Result:

```
csv      38.0 MB
json     115.0 MB
parquet  7.6 MB
orc      3.9 MB
avro     13.0 MB
```

</details>

<details>
<summary><b>A2</b> &nbsp;Read the CSV back without a schema. What type does every column get? And how many jobs does inferSchema start ...</summary>

**Every column is a string**, because CSV is plain text without types. Reading the header is 1 job; with `inferSchema`, Spark reads the whole file once more just to guess the types (2 jobs), and your query will read it again. That is why CSV is called heavy: big files, no types, every value must be parsed, and no way to skip columns or rows. When you must read CSV, give the schema yourself (`.schema(...)`).

```python
spark.read.option("header", True).csv("data/shop/formats/csv").printSchema()
print(count_jobs(lambda: spark.read.option("header", True).csv("data/shop/formats/csv")), "job without inferSchema")
print(count_jobs(lambda: spark.read.option("header", True).option("inferSchema", True).csv("data/shop/formats/csv")), "jobs with inferSchema")
```

Result:

```
root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- quantity: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- status: string (nullable = true)

1 job without inferSchema
2 jobs with inferSchema
```

</details>

<details>
<summary><b>A3</b> &nbsp;Sum the quantity of each format and time it with best_time. Which formats are the fastest to read?</summary>

**Parquet, several times faster than CSV and JSON** (ORC and Avro land in between here). Parquet reads only the bytes of the `quantity` column and decodes numbers directly; the text formats must read every line and parse it. With a wide table (say 100 columns) the difference grows much bigger.

```python
for fmt in ["csv", "json", "parquet", "orc", "avro"]:
    df = spark.read.format(fmt).schema(big_orders.schema).option("header", True).load(f"data/shop/formats/{fmt}")
    print(f"{fmt:8}", best_time(lambda: df.agg(F.sum("quantity")).first()), "s")
```

Result:

```
csv      0.16 s
json     0.26 s
parquet  0.04 s
orc      0.04 s
avro     0.08 s
```

*Your times will differ.*

</details>

<details>
<summary><b>A4</b> &nbsp;Avro keeps the schema inside each file. Print the first bytes of one Avro file: what do you see?</summary>

**The schema, as JSON** (`avro.schema ... {"type":"record", ... "fields": [...]}`), followed by the records one after another. Every record is self-contained and the schema travels with the data, so Avro suits streams such as Kafka: a producer can write one message at a time (Parquet needs a big batch of rows to build its columns), and readers can handle a changed schema with Avro's compatibility rules, often with a schema registry.

```python
import glob
avro_file = glob.glob("data/shop/formats/avro/*.avro")[0]
print(open(avro_file, "rb").read(400))
```

Result:

```
b'Obj\x01\x06\x16avro.schema\x90\x05{"type":"record","name":"topLevelRecord","fields":[{"name":"order_id","type":"long"},{"name":"customer_id","type":["long","null"]},{"name":"product_id","type":["long","null"]},{"name":"quantity","type":["long","null"]},{"name":"order_date","type":[{"type":"int","logicalType":"date"},"null"]},{"name":"status","type":"string"}]}0org.apache.spark.version\n4.2.0\x14avro.codec\x0csnappy\x00&\xbc}'
```

</details>

<details>
<summary><b>A5</b> &nbsp;Parquet stores the min and max of every column for each row group (a block of rows). Write big_orders once ...</summary>

**None in the unsorted file, most of them in the sorted one.** Spark pushes `quantity = 5` down to the Parquet reader (`PushedFilters: [IsNotNull(quantity), EqualTo(quantity,5)]` in the plan), and the reader skips every row group whose min to max range cannot contain 5. In the unsorted file every row group holds quantities 1 to 5, so nothing can be skipped. **Predicate pushdown** pays off when the data is laid out so that the ranges are narrow: sort (or Z-order, Lab 09) by the columns you filter on.

```python
import glob
import pyarrow.parquet as pq
big_orders.coalesce(1).write.mode("overwrite").option("parquet.block.size", 256 * 1024).parquet("data/shop/unsorted")
(big_orders.orderBy("quantity").coalesce(1)
           .write.mode("overwrite").option("parquet.block.size", 256 * 1024).parquet("data/shop/sorted"))

def quantity_ranges(folder):
    """(min, max) of quantity in every row group of the folder's Parquet file, from the file footer."""
    meta = pq.ParquetFile(glob.glob(f"{folder}/*.parquet")[0]).metadata
    col = meta.schema.names.index("quantity")
    return [(meta.row_group(i).column(col).statistics.min, meta.row_group(i).column(col).statistics.max)
            for i in range(meta.num_row_groups)]

print(quantity_ranges("data/shop/unsorted")[:5])
print(quantity_ranges("data/shop/sorted")[:5])
for folder in ["data/shop/unsorted", "data/shop/sorted"]:
    ranges = quantity_ranges(folder)
    skippable = sum(1 for low, high in ranges if not low <= 5 <= high)
    print(folder, "->", skippable, "of", len(ranges), "row groups can be skipped")
spark.read.parquet("data/shop/sorted").filter(F.col("quantity") == 5).explain()
```

Result:

```
[(1, 5), (1, 5), (1, 5), (1, 5), (1, 5)]
[(1, 1), (1, 1), (1, 1), (1, 1), (1, 1)]
data/shop/unsorted -> 0 of 184 row groups can be skipped
data/shop/sorted -> 147 of 184 row groups can be skipped
   +- FileScan parquet [order_id#429L,customer_id#430L,product_id#431L,quantity#432L,order_date#433,status#434] Batched: true, DataFilters: [isnotnull(quantity#432L), (quantity#432L = 5)], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/..., PartitionFilters: [], PushedFilters: [IsNotNull(quantity), EqualTo(quantity,5)], ReadSchema: struct<order_id:bigint,customer_id:bigint,product_id:bigint,quantity:bigint,order_date:date,statu...
```

</details>

<details>
<summary><b>A6</b> &nbsp;Write big_orders partitioned by status, then filter status = 'cancelled'. What does the plan show under ...</summary>

**`PartitionFilters: [isnotnull(status), (status = cancelled)]`, and only the files of one folder** (4 of 12). The value of a partition column is in the folder name (`status=cancelled/`), so Spark skips whole folders without opening a file (**partition pruning**). Partition by a column you often filter on that has few values (a date, a country), never by something like order_id, which would create millions of folders.

```python
big_orders.write.mode("overwrite").partitionBy("status").parquet("data/shop/by_status")
cancelled = spark.read.parquet("data/shop/by_status").filter(F.col("status") == "cancelled")
cancelled.explain()
all_files = spark.read.parquet("data/shop/by_status").select("_metadata.file_path").distinct().count()
read_files = cancelled.select("_metadata.file_path").distinct().count()
print(read_files, "of", all_files, "files read")
```

Result:

```
+- FileScan parquet [order_id#436L,customer_id#437L,product_id#438L,quantity#439L,order_date#440,status#441] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/..., PartitionFilters: [isnotnull(status#441), (status#441 = cancelled)], PushedFilters: [], ReadSchema: struct<order_id:bigint,customer_id:bigint,product_id:bigint,quantity:bigint,order_date:date>
4 of 12 files read
```

</details>

<details>
<summary><b>A7</b> &nbsp;Write big_orders partitioned by order_date twice: after repartition(50) and after repartition("order_date"). ...</summary>

**About 4,550 files against 91, and the small files are several times slower to read** (with far more read tasks). Each of the 50 write tasks held rows of every day, so each wrote one file into each of the 91 folders: 50 × 91 tiny files. Every file costs a listing, an open and a footer read, and Spark schedules work per file: that is the **small file problem**. Fix it by repartitioning by the partition column before writing (one file per folder), by writing fewer, bigger files, or by compacting (`OPTIMIZE` in Delta, Lab 09).

```python
big_orders.repartition(50).write.mode("overwrite").partitionBy("order_date").parquet("data/shop/small_files")
big_orders.repartition("order_date").write.mode("overwrite").partitionBy("order_date").parquet("data/shop/one_file_per_day")
import glob
for folder in ["data/shop/small_files", "data/shop/one_file_per_day"]:
    files = len(glob.glob(f"{folder}/*/*.parquet"))
    df = spark.read.parquet(folder)
    print(f"{folder}: {files} files, {df.rdd.getNumPartitions()} read tasks, "
          f"{best_time(lambda: df.agg(F.sum('quantity')).first())} s")
```

Result:

```
data/shop/small_files: 4550 files, 143 read tasks, 0.82 s
data/shop/one_file_per_day: 91 files, 4 read tasks, 0.04 s
```

*Your times will differ.*

</details>

<details>
<summary><b>A8</b> &nbsp;Write big_orders as one plain CSV file and as one gzip CSV file. With spark.sql.files.maxPartitionBytes set ...</summary>

**5 for the plain file, 1 for the gzip file.** Spark can cut a plain CSV file into 8 MB pieces and read them in parallel, because a reader can start in the middle and look for the next line break (Parquet, ORC and Avro split too). A gzip stream can only be decompressed from its beginning, so the whole file is one task: it is **not splittable**. A big gzip file makes one slow task while the other cores wait; use bzip2 (splittable) or, better, Parquet.

```python
big_orders.coalesce(1).write.mode("overwrite").option("header", True).csv("data/shop/csv_plain")
big_orders.coalesce(1).write.mode("overwrite").option("header", True).option("compression", "gzip").csv("data/shop/csv_gzip")
spark.conf.set("spark.sql.files.maxPartitionBytes", 8 * 1024 * 1024)      # 8 MB per read partition
for folder in ["data/shop/csv_plain", "data/shop/csv_gzip"]:
    print(folder, spark.read.option("header", True).csv(folder).rdd.getNumPartitions(), "read partitions")
spark.conf.unset("spark.sql.files.maxPartitionBytes")
```

Result:

```
data/shop/csv_plain 5 read partitions
data/shop/csv_gzip 1 read partitions
```

</details>

<details>
<summary><b>A9</b> &nbsp;Write 5 orders without and 5 orders with a new column discount into two folders of data/shop/evolve. Read the ...</summary>

**Without `mergeSchema`, Spark takes the schema from one file, so `discount` is missing; with it, you get the columns of all files, and the old rows show `discount` as null.** That is **schema evolution**: you can add columns over time without rewriting old files. (The folder names `batch=1` and `batch=2` became a column too: partition discovery.)

```python
big_orders.filter("order_id <= 5").write.mode("overwrite").parquet("data/shop/evolve/batch=1")
(big_orders.filter("order_id > 5 AND order_id <= 10").withColumn("discount", F.lit(0.1))
           .write.mode("overwrite").parquet("data/shop/evolve/batch=2"))
print(spark.read.parquet("data/shop/evolve").columns)
merged = spark.read.option("mergeSchema", "true").parquet("data/shop/evolve")
print(merged.columns)
merged.orderBy("order_id").show()
```

Result:

```
['order_id', 'customer_id', 'product_id', 'quantity', 'order_date', 'status', 'batch']
['order_id', 'customer_id', 'product_id', 'quantity', 'order_date', 'status', 'discount', 'batch']
+--------+-----------+----------+--------+----------+---------+--------+-----+
|order_id|customer_id|product_id|quantity|order_date|   status|discount|batch|
+--------+-----------+----------+--------+----------+---------+--------+-----+
|       1|       2647|       106|       1|2024-02-25|  shipped|    NULL|    1|
|       2|        117|       104|       1|2024-01-07|cancelled|    NULL|    1|
|       3|       3555|       107|       4|2024-03-10|cancelled|    NULL|    1|
|       4|       2297|       107|       3|2024-03-30|cancelled|    NULL|    1|
|       5|       2501|       102|       3|2024-01-14|  shipped|    NULL|    1|
|       6|       2880|       106|       5|2024-03-15|delivered|     0.1|    2|
|       7|       3823|       104|       1|2024-01-27|  shipped|     0.1|    2|
|       8|       3252|       103|       2|2024-03-06|delivered|     0.1|    2|
|       9|       3655|       106|       5|2024-01-01|delivered|     0.1|    2|
|      10|       2899|       102|       3|2024-03-23|delivered|     0.1|    2|
+--------+-----------+----------+--------+----------+---------+--------+-----+
```

</details>

<details>
<summary><b>A10</b> &nbsp;Parquet files can't be changed. To fix the status of a single order, how much data must you write?</summary>

**All of it**, to change one value. A Parquet file is written once and never modified: there is no "update one row". To change data you rewrite whole files, here into a new folder (overwriting the folder you are reading from would delete your input before Spark reads it). That is Parquet's main disadvantage, together with slow writes for tiny batches. Delta Lake (Lab 09) adds updates and deletes on top of Parquet: it rewrites only the files that contain changed rows and records the change in a log.

```python
fixed = (spark.read.parquet("data/shop/formats/parquet")
              .withColumn("status", F.when(F.col("order_id") == 5, "returned").otherwise(F.col("status"))))
fixed.write.mode("overwrite").parquet("data/shop/formats/parquet_fixed")
print(folder_mb("data/shop/formats/parquet"), "MB of data,", folder_mb("data/shop/formats/parquet_fixed"), "MB written to change one value")
```

Result:

```
7.6 MB of data, 7.6 MB written to change one value
```

</details>

<details>
<summary><b>B1</b> &nbsp;Turn review_date (text like 15/01/2024) into a real date column review_day. Show review_id, review_date and ...</summary>

```python
reviews.select("review_id", "review_date", F.to_date("review_date", "dd/MM/yyyy").alias("review_day")).show()
```

</details>

<details>
<summary><b>B2</b> &nbsp;For the orders placed in January and February, show order_id, order_date, the last day of that month ...</summary>

```python
(orders.filter(F.month("order_date") <= 2)
       .select("order_id", "order_date",
               F.last_day("order_date").alias("month_end"),
               F.datediff(F.last_day("order_date"), "order_date").alias("days_left"))
       .orderBy("order_id")
       .show())
```

</details>

<details>
<summary><b>B3</b> &nbsp;Count the orders per week: show week_start (the Monday of the week, as a date) and orders, for the weeks that ...</summary>

```python
(orders.groupBy(F.date_trunc("week", "order_date").cast("date").alias("week_start"))
       .count()
       .withColumnRenamed("count", "orders")
       .orderBy("week_start")
       .show())
```

</details>

<details>
<summary><b>B4</b> &nbsp;Show the quarter and the month name of each customer's signup date: name, signup_date, quarter and month_name.</summary>

```python
customers.select("name", "signup_date",
                 F.quarter("signup_date").alias("quarter"),
                 F.date_format("signup_date", "MMMM").alias("month_name")).show()
```

</details>

<details>
<summary><b>B5</b> &nbsp;Clean the comments: remove the spaces at both ends, make them lower case, and remove every character that is ...</summary>

```python
(reviews.select("review_id",
                 F.regexp_replace(F.lower(F.trim("comment")), "[^a-z0-9 ]", "").alias("clean_comment"))
        .show(truncate=False))
```

</details>

<details>
<summary><b>B6</b> &nbsp;How many words does each comment have? Show review_id and words.</summary>

```python
reviews.select("review_id", F.size(F.split(F.trim("comment"), r"\s+")).alias("words")).show()
```

</details>

<details>
<summary><b>B7</b> &nbsp;Which reviews mention "slow" in their comment, in any letter case? Show review_id and comment.</summary>

```python
reviews.filter(F.lower("comment").contains("slow")).select("review_id", "comment").show(truncate=False)
```

</details>

<details>
<summary><b>B8</b> &nbsp;Split the tags into an array and show one row per review and tag (reviews without tags drop out). Show ...</summary>

```python
reviews.select("review_id", F.explode(F.split("tags", ";")).alias("tag")).show()
```

</details>

<details>
<summary><b>B9</b> &nbsp;How often is each tag used? Show tag and reviews, most used first; on a tie, A to Z.</summary>

```python
(reviews.select(F.explode(F.split("tags", ";")).alias("tag"))
        .groupBy("tag").count()
        .withColumnRenamed("count", "reviews")
        .orderBy(F.col("reviews").desc(), "tag")
        .show())
```

</details>

<details>
<summary><b>B10</b> &nbsp;For each customer, list the different products they ordered as an array of product names, sorted A to Z. Show ...</summary>

```python
(orders.filter(F.col("customer_id").isNotNull())
       .join(products, "product_id")
       .groupBy("customer_id")
       .agg(F.array_sort(F.collect_set("product_name")).alias("products"))
       .orderBy("customer_id")
       .show(truncate=False))
```

</details>

<details>
<summary><b>B11</b> &nbsp;Collect each customer's product_ids into a sorted array, then keep only the customers whose array contains ...</summary>

```python
(orders.filter(F.col("customer_id").isNotNull())
       .groupBy("customer_id")
       .agg(F.array_sort(F.collect_set("product_id")).alias("product_ids"))
       .filter(F.array_contains("product_ids", 101))
       .orderBy("customer_id")
       .show(truncate=False))
```

</details>

<details>
<summary><b>B12</b> &nbsp;Show the number of orders on every day from 2024-03-01 to 2024-03-10, with 0 on the days without orders. ...</summary>

```python
days = spark.range(1).select(
    F.explode(F.sequence(F.to_date(F.lit("2024-03-01")), F.to_date(F.lit("2024-03-10")))).alias("day"))
per_day = orders.groupBy(F.col("order_date").alias("day")).count()
(days.join(per_day, "day", "left")
     .select("day", F.coalesce(F.col("count"), F.lit(0)).alias("orders"))
     .orderBy("day")
     .show())
```

</details>

<details>
<summary><b>B13</b> &nbsp;Mask the emails: keep the first letter, then *, then everything from the @ on (like a*@mail.uz). Customers ...</summary>

```python
(customers.filter(F.col("email").isNotNull())
          .select("name",
                  F.concat(F.substring("email", 1, 1), F.lit("***@"), F.substring_index("email", "@", -1)).alias("masked_email"))
          .show())
```

</details>